**Walmart Data Engineer Interview questions**

---

### ✅ **SQL**
- Write a query to find the 2nd highest salary  
- Difference between **RANK** and **DENSE_RANK**  
- Explain **window functions** with example  
- Query to find **duplicate records**

---

In [0]:
%sql
SELECT MAX(salary) AS second_highest_salary
FROM employees
WHERE salary < (SELECT MAX(salary) FROM employees);


| Function | Behavior | Example |
| --- | --- | --- |
| ``RANK()`` | Skips ranks after ties | 1, 2, 2, 4 |
| ``DENSE_RANK()`` | No gaps after ties | 1, 2, 2, 3 |

In [0]:
%sql
SELECT name, salary, RANK() OVER (ORDER BY salary DESC) AS rank
FROM employees;


✅ Explanation: Window functions operate on a set of rows related to the current row without collapsing them.

In [0]:
%sql
SELECT name, COUNT(*) 
FROM employees 
GROUP BY name 
HAVING COUNT(*) > 1;


### ✅ **PySpark and Python**
- Explain **lazy evaluation** in Spark  
- What is a **DAG** in Spark?  
- **Repartition vs Coalesce**  
- How do you handle **skewed data**?

---

**1. Lazy evaluation:**  
Spark builds a logical plan first and executes only when an action (like `count()` or `collect()`) is called — improving optimization.


**2. DAG (Directed Acyclic Graph):**  
Represents the sequence of transformations. Spark scheduler uses DAGs to determine execution stages.

| Operation | Increases partitions | Decreases partitions | Shuffle |
| --- | --- | --- | --- |
| ``repartition()`` | ✅ | ✅ | Yes |
| ``coalesce()`` | ❌ | ✅ | No (efficient) |

**4. Handling skewed data:**  
Use techniques like *salting keys*, *broadcast joins*, or *repartitioning* based on data distribution.

%md
### ✅ **Data Architecture**
- What is **SCD Type 2**? Implement it 
- Explain **Medallion Architecture**  
- **Delta Lake vs Parquet** difference  
- What is **MERGE** in Delta Lake?

---

**1. SCD Type 2:**  
Maintains history by adding new rows with versioning and timestamps.  
Example columns: `start_date`, `end_date`, `is_current`.

In [0]:
# SCD Type 2 Implementation



from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp, lit

# Assume 'source_df' contains new data, 'target_table' is the Delta table name

delta_table = DeltaTable.forName(spark, "target_table")

# Mark existing records as not current if matching keys and values have changed
update_condition = "t.id = s.id AND t.is_current = true AND (t.column1 <> s.column1 OR t.column2 <> s.column2)"

delta_table.alias("t").merge(
    source_df.alias("s"),
    "t.id = s.id"
).whenMatchedUpdate(
    condition=update_condition,
    set={
        "end_date": "current_timestamp()",
        "is_current": "false"
    }
).whenNotMatchedInsert(
    values={
        "id": "s.id",
        "column1": "s.column1",
        "column2": "s.column2",
        "start_date": "current_timestamp()",
        "end_date": "null",
        "is_current": "true"
    }
).execute()

**2. Medallion Architecture:**  
- **Bronze:** Raw data  
- **Silver:** Cleaned and enriched  
- **Gold:** Aggregated and business-ready  

3. Delta Lake vs Parquet

| Feature | Delta Lake | Parquet |
| --- | --- | --- |
| ACID Transactions | ✅ | ❌ |
| Schema Evolution | ✅ | Limited |
| Time Travel | ✅ | ❌ |

In [0]:
4. MERGE in Delta Lake:  
Used for upserts (insert/update) based on matching conditions.



# Implement Upsert condition

# Assuming 'target_table' is the Delta table and 'updates_df' is the Spark DataFrame with updates

from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "target_table")

delta_table.alias("t").merge(
    updates_df.alias("u"),
    "t.id = u.id"  # Replace 'id' with your unique key column
).whenMatchedUpdate(
    {
        "column1": "u.column1",
        "column2": "u.column2",
        # Add more columns as needed
    }
).whenNotMatchedInsertAll() \
 .execute()


### ✅ **Cloud and Tools**
- Which **cloud platforms** have you worked on?  
- How does **Auto Loader** work?  
- **ADF pipeline vs Databricks job**

---

**1. Cloud platforms:** Azure, AWS, GCP — mention your hands-on experience.  

In [0]:
2. Auto Loader: Incrementally loads new files from cloud storage into Delta tables.


from pyspark.sql.functions import *

# Configure Auto Loader for incremental load from cloud storage
df = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "/mnt/schema_location") \
    .load("/mnt/input_path")

# Write incrementally to Delta table
df.writeStream.format("delta") \
    .option("checkpointLocation", "/mnt/checkpoint_location") \
    .outputMode("append") \
    .start("/mnt/delta_table_path")

**3. ADF pipeline vs Databricks job:**  
ADF orchestrates workflows; Databricks executes Spark transformations.

### ✅ **Scenario Based**
- Your pipeline failed at 2 AM. What do you do?  
- How do you **optimize a slow Spark job**?  
- Design an **end-to-end ETL pipeline**

---

**1. Pipeline failed at 2 AM:**  
Check logs, identify root cause, rerun failed tasks, and implement alerting for future prevention.

**2. Optimize slow Spark job:**  
- Use partition pruning  
- Cache intermediate results  
- Avoid shuffles  
- Tune executor memory and cores

**3. End-to-end ETL pipeline design:**  
Source → Ingest (ADF/Auto Loader) → Transform (Databricks) → Store (Delta Lake) → Serve (Power BI/Synapse)